# Lab 4.2 – CI Quality Gate

Run the evaluation suite headlessly, inspect metrics JSON, and demonstrate that an unrealistically high threshold fails the gate.

In [ ]:
# --- Environment bootstrap (Colab + local + Docker) ---
import os, sys
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

REPO_URL = "https://github.com/fischer3-net/accurate_secure_rag_systems.git"
LAB_DIR = "labs/04-evaluation"

if _in_colab():
    REPO_ROOT = Path("/content/accurate_secure_rag_systems")
    if not REPO_ROOT.exists():
        get_ipython().system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    LAB = REPO_ROOT / LAB_DIR
    os.chdir(LAB)
    sys.path.insert(0, str(LAB))
    sys.path.insert(0, str(REPO_ROOT / "labs" / "01-chunking"))
    sys.path.insert(0, str(REPO_ROOT / "labs" / "03-skills"))
    get_ipython().run_line_magic("pip", "install -q pydantic python-dotenv langchain-text-splitters langchain-core pytest pyyaml pandas")
    print("Colab ready | LAB =", LAB)
else:
    LAB = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(LAB) not in sys.path:
        sys.path.insert(0, str(LAB))
    print("Local ready | LAB =", LAB)


In [ ]:
import json, sys
from pathlib import Path

LAB = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(LAB))

from src.eval_runner import run_evaluation

GOLDEN = LAB / "data" / "golden_dataset.jsonl"
FIXTURES = LAB / "data" / "fixtures"
CORPUS = LAB / "data" / "rag_chunks.jsonl"
OUT = LAB / "output" / "metrics.json"
OUT.parent.mkdir(exist_ok=True)

In [ ]:
report = run_evaluation(
    golden_path=GOLDEN,
    fixtures_dir=FIXTURES,
    corpus_path=CORPUS,
    min_mean_overall=0.50,
    min_mean_control_hit_rate=0.40,
)
OUT.write_text(json.dumps(report, indent=2))
print(json.dumps({k: report[k] for k in report if k != "row_scores"}, indent=2))
print("Wrote", OUT)

In [ ]:
strict = run_evaluation(
    golden_path=GOLDEN,
    fixtures_dir=FIXTURES,
    corpus_path=CORPUS,
    min_mean_overall=0.99,
    min_mean_control_hit_rate=0.99,
)
print("Strict thresholds_passed:", strict["thresholds_passed"])
print("Messages:", strict["messages"])

## CI wiring

- GitHub Actions: `.github/workflows/eval.yml`
- Optional Cloud Build: `scripts/cloudbuild-eval.yaml`

Thresholds live in `src/thresholds.yaml` and in the workflow flags. Document any changes you make for the Capstone.